In [0]:
from pyspark.sql.functions import lit, col, initcap, concat, trim, regexp_replace, lower, upper, split, when, substr, length, coalesce, to_date, Column
from pyspark.sql.types import *

In [0]:
# clean customer full name
def clean_name(col_name: Column | str) -> Column:
    only_alpha = regexp_replace(col_name, "[^a-zA-Z\\s]", "")
    return trim(initcap(only_alpha))

# clean and upper text
def clean_text(col_name: Column | str) -> Column:
    return upper(trim(col_name))

# Clean formatted customer emirates id (eid)
def clean_emirates_id(col_name: Column | str) -> Column:
    digit_only = regexp_replace(trim(col_name), "[^0-9]","")
    formatted_eid = regexp_replace(
        digit_only,
        "^([0-9]{3})([0-9]{4})([0-9]{7})([0-9]{1})",
        "$1-$2-$3-$4"
    )
    return formatted_eid
# clean email
def clean_email(col_name: Column | str) -> Column:
    clean_email = regexp_replace(lower(trim(col_name)), "[^a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2]", "")
    return clean_email

# clean and formatted phone
def clean_phone(col_name: Column | str) -> Column:
    digit_only = regexp_replace(trim(col_name), "[^0-9]","")

    init_digit = when(digit_only.startswith("0"), regexp_replace(digit_only, "^0","971")).otherwise(digit_only)

    formatted_phone = regexp_replace(
        init_digit,
        "^([0-9]{3})([0-9]{2})([0-9]{3})([0-9]{4})$",
        "+$1-$2-$3-$4"
    )
    return formatted_phone

# clean nationality
def clean_nationality(col_name: Column | str) -> Column:
    cleaned = trim(col_name)
    return when(length(cleaned) <=3, upper(cleaned)).otherwise(initcap(lower(cleaned)))


"""
    Cleans dirty string date column and converts it into standard PySpark DateType (YYYY-MM-DD).
    Handles multiple date patterns: yyyy-MM-dd, dd/MM/yyyy, MM-dd-yyyy, yyyy/MM/dd
    """
def parse_mixed_date(col_name: Column | str) -> Column:
    cleaned = trim(col_name)
    return coalesce(
        to_date(cleaned, "yyyy-MM-dd"),   # e.g., 1991-01-17
        to_date(cleaned, "dd/MM/yyyy"),   # e.g., 17/01/1991
        to_date(cleaned, "MM-dd-yyyy"),   # e.g., 01-17-1991
        to_date(cleaned, "yyyy/MM/dd"),   # e.g., 1991/01/17
        to_date(cleaned)
    )